In [326]:
from pathlib import Path
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader
import re

In [327]:
def load_data(folder_path):
    data_list = []
    # Loop through all items in the folder
    for file_path in folder_path.iterdir():
        # Check if the item is a file (skips subfolders)
        if file_path.is_file():
            if(file_path.suffix == ".c"):
                label = "c"
            elif(file_path.suffix == ".java"):
                label = "java"
            else:
                continue   
            content = file_path.read_text(encoding="utf-8")
            data_list.append((content, label))

    return data_list


In [328]:
folder_path = Path("dataset")
data = load_data(folder_path)

In [329]:
output = []
dictionary = {}
i = 0

for item in data:
    output += (re.split(r'([{}()\[\];,=+\-*/<>!])|\s+',item[0]))
for word in output:
    # remove the noises
    if word =="" or word in dictionary:
        continue
    dictionary[word] = i
    i+=1

In [330]:
def text_to_bow(text, dictionary):
    Bag_of_Words = [0] * len(dictionary)
    
    output = re.split(r'([{}()\[\];,=+\-*/<>!])|\s+',text)
    for word in output:
        if word in dictionary:
            Bag_of_Words[dictionary[word]] +=1

    Bag_of_Words_tensor = torch.tensor(Bag_of_Words, dtype=torch.float32)
    
    return Bag_of_Words_tensor

In [331]:
class CodeDataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
        
    def __len__(self):
        return len(self.data_list)
        
    def __getitem__(self, idx):
        Bag_of_Words_tensor = text_to_bow(self.data_list[idx][0], dictionary)
        label = 1 if (self.data_list[idx][1] == 'c') else 0

        return Bag_of_Words_tensor, torch.tensor(label, dtype=torch.long)

In [332]:
dataset = CodeDataset(data)
dataload =  DataLoader(dataset, batch_size=4, shuffle=True)

In [333]:
class CodeClassifier(nn.Module):
    def __init__(self,input_size):
        super().__init__()
        hidden_size = 128
        num_classes = 2
        self.l1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.l2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self,x):
        output = self.l1(x)
        output = self.activation(output)
        output = self.l2(output)

        return output


In [334]:
vocab_size = len(dictionary)
model = CodeClassifier(input_size=vocab_size)
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 20

In [335]:
for epoch in range(epochs):
    for inputs, labels in dataload:

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward() 
        
        optimizer.step()
        
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
    

Epoch [1/20], Loss: 0.2371
Epoch [2/20], Loss: 0.0001
Epoch [3/20], Loss: 0.0184
Epoch [4/20], Loss: 0.0277
Epoch [5/20], Loss: 2.1867
Epoch [6/20], Loss: 0.0944
Epoch [7/20], Loss: 0.2662
Epoch [8/20], Loss: 0.0599
Epoch [9/20], Loss: 0.0028
Epoch [10/20], Loss: 0.2069
Epoch [11/20], Loss: 0.0000
Epoch [12/20], Loss: 0.0675
Epoch [13/20], Loss: 0.0140
Epoch [14/20], Loss: 0.2548
Epoch [15/20], Loss: 0.0550
Epoch [16/20], Loss: 0.0002
Epoch [17/20], Loss: 0.0095
Epoch [18/20], Loss: 0.0000
Epoch [19/20], Loss: 0.0052
Epoch [20/20], Loss: 0.0000


In [336]:
def predict(test_code,dictionary,model):
    Bag_of_Words_tensor = text_to_bow(test_code,dictionary)
    Bag_of_Words_tensor = Bag_of_Words_tensor.unsqueeze(0)
    logits = model(Bag_of_Words_tensor)
    pred = torch.argmax(logits, dim=1)
    return pred.item()
        

In [337]:
def decide(input):
    print("c") if (input == 1) else print("java")

In [338]:

    
test_code_c = """#include <stdio.h>
#include <stdlib.h>

void update_value(int *ptr) {
    *ptr = *ptr + 10;
}

int main() {
    int x = 5;
    int *ptr = &x;
    
    printf("Before: %d\n", x);
    update_value(ptr);
    printf("After: %d\n", x);
    
    int *arr = malloc(3 * sizeof(int));
    arr[0] = 10;
    arr[1] = 20;
    free(arr);
    
    return 0;
}"""
test_code_java ="""
    public Book(String id, String title, String author, String category, int year, int copies) 
{ 
    this.ID = id; 
    this.Title = title; 
    this.Author = author; 
    this.Category = category; 
    this.Year = year; 
    this.Copies = copies;
    public String getID() { 
        return ID; 
    }
    
    public void setID(String ID) { 
        if (ID != null && !ID.trim().isEmpty()) {
            this.ID = ID; 
        }
    }

    public String getName() { 
        return Name; 
    }
    
    public void setName(String n) { 
        if (n != null && !n.trim().isEmpty()) {
            this.Name = n; 
        }
    }

    public String getEmail() { 
        return Email; 
    }"""

decide(predict(test_code_c,dictionary,model))
decide(predict(test_code_java,dictionary,model))


c
java
